## Kaggle Homework- Pranika Chandra-@PranikaC

Three inspiring datasets:
1. https://www.kaggle.com/code/arshikhan810/ensemble-using-tpu 
2. https://www.kaggle.com/code/mohit98765/medlit-gemma4-medical-literacy-assistant
3. https://www.kaggle.com/code/hbistechie/ps6-e4-irrigation-prediction-lgbm

## Exploratory Data Analysis

Importing required packages prior to performing Exploratory Data Analysis

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sweetviz as sv
from sklearn.model_selection import train_test_split # for boosting/bagging modeling
from xgboost import XGBClassifier # for boosting/bagging modeling
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score # for boosting/bagging modeling
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
import optuna
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

c:\Users\prchandr\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Import + View the training dataset 

In [6]:
df_train = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\train.csv')
df_test = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\test.csv')

In [7]:
X = df_train.drop('Irrigation_Need', axis=1)
y = df_train['Irrigation_Need'].copy()

In [8]:
df_train.head(5)

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


Performing Exploratory Data Analysis using SweetViz

In [19]:
report = sv.analyze(df_train)
report.show_html()

Done! Use 'show' commands to display/save.   |██████████| [100%]   00:01 -> (00:00 left)


Report SWEETVIZ_REPORT.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


What did the EDA tell us?

What insights did you gain? Is there a feature you think is especially important? Any potential issues?

Based on the EDA, we can see that there are 630,000 null values and that there are no strong direct relationships numerically nor categorically. Around 60% of the data represents low irrigation_need, whereas less than 10% of the data shows high irrigation need. The features look evenly distributed and therefore shows no clear, contributing features.

## Gradient Boosting Model

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

best_model = GradientBoostingClassifier(
    n_estimators=10,
    learning_rate=0.1,
    max_depth=2,
    random_state=42
)

print("Training final model...")
best_model.fit(X_train, y_train)
print("Training complete.")

y_pred = best_model.predict(X_valid)
print(classification_report(y_valid, y_pred))

test_pred = best_model.predict(X_test_final)

submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': test_pred
})

submission.to_csv('GradientBoosting_submission.csv', index=False)
print("Saved.")

Training final model...
Training complete.
              precision    recall  f1-score   support

        High       0.80      0.30      0.43      4202
         Low       0.76      0.94      0.84     73983
      Medium       0.78      0.53      0.63     47815

    accuracy                           0.76    126000
   macro avg       0.78      0.59      0.63    126000
weighted avg       0.77      0.76      0.75    126000

Saved.


# Gradient Boosting Model (with feature engineering and tuning)

In [72]:
import pandas as pd
import numpy as np

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

# =========================================
# LOAD DATA
# =========================================
df_train = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\train.csv')
df_test = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\test.csv')

# =========================================
# FEATURES / TARGET
# =========================================
X = df_train.drop(columns=['id', 'Irrigation_Need', 'Previous_Irrigation_mm']).copy()
y = df_train['Irrigation_Need'].copy()

X_test = df_test.drop(columns=['id', 'Previous_Irrigation_mm']).copy()
test_ids = df_test['id'].copy()

# =========================================
# FEATURE ENGINEERING
# =========================================
X['Moisture_Temp'] = X['Soil_Moisture'] * X['Temperature_C']
X_test['Moisture_Temp'] = X_test['Soil_Moisture'] * X_test['Temperature_C']

X['Rain_Humidity'] = X['Rainfall_mm'] * X['Humidity']
X_test['Rain_Humidity'] = X_test['Rainfall_mm'] * X_test['Humidity']

X['Sun_Temp'] = X['Sunlight_Hours'] * X['Temperature_C']
X_test['Sun_Temp'] = X_test['Sunlight_Hours'] * X_test['Temperature_C']

X['Moisture_Rain_Ratio'] = X['Soil_Moisture'] / (X['Rainfall_mm'] + 1)
X_test['Moisture_Rain_Ratio'] = X_test['Soil_Moisture'] / (X_test['Rainfall_mm'] + 1)

X['Temp_Humidity_Ratio'] = X['Temperature_C'] / (X['Humidity'] + 1)
X_test['Temp_Humidity_Ratio'] = X_test['Temperature_C'] / (X_test['Humidity'] + 1)

# =========================================
# CLEAN NUMERIC ISSUES
# =========================================
X = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

num_cols = X.select_dtypes(include=[np.number]).columns
for col in num_cols:
    median_value = X[col].median()
    X[col] = X[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

# =========================================
# ONE-HOT ENCODING
# =========================================
X = pd.get_dummies(X, drop_first=False)
X_test = pd.get_dummies(X_test, drop_first=False)

X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

# =========================================
# FASTER CV
# =========================================
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

# =========================================
# SMALL PARAMETER SEARCH
# =========================================
param_list = [
    {'n_estimators': 250, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 0.8},
    {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 4, 'subsample': 0.8},
    {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 3, 'subsample': 0.9}
]

best_score = -np.inf
best_params = None

for params in param_list:
    print(f"Trying params: {params}")

    model = GradientBoostingClassifier(
        n_estimators=params['n_estimators'],
        learning_rate=params['learning_rate'],
        max_depth=params['max_depth'],
        subsample=params['subsample'],
        random_state=42
    )

    scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring='f1_weighted',
        n_jobs=-1
    )

    mean_score = scores.mean()
    print(f"Mean CV Weighted F1: {mean_score:.6f}\n")

    if mean_score > best_score:
        best_score = mean_score
        best_params = params

print("Best Params:", best_params)
print("Best CV Weighted F1:", best_score)

# =========================================
# TRAIN FINAL MODEL
# =========================================
final_model = GradientBoostingClassifier(
    n_estimators=best_params['n_estimators'],
    learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'],
    subsample=best_params['subsample'],
    random_state=42
)

print("Training final model...")
final_model.fit(X, y)
print("Training complete.")

Trying params: {'n_estimators': 250, 'learning_rate': 0.05, 'max_depth': 3, 'subsample': 0.8}
Mean CV Weighted F1: 0.984886

Trying params: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 4, 'subsample': 0.8}
Mean CV Weighted F1: 0.985668

Trying params: {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 3, 'subsample': 0.9}
Mean CV Weighted F1: 0.984438

Best Params: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 4, 'subsample': 0.8}
Best CV Weighted F1: 0.9856678501610878
Training final model...
Training complete.


In [74]:
# =========================================
# PREDICT TEST
# =========================================
test_pred = final_model.predict(X_test)

submission = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': test_pred
})

submission.to_csv(r'C:\Users\prchandr\Downloads\UpdatedGradientBoostingsubmission.csv', index=False)
print("Saved.")

Saved.


# Random Forest Model

In [67]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10, 
    random_state=42,
    n_jobs=-1
)

print("Training Random Forest...")
rf_model.fit(X_train, y_train)
print("Training complete.")

# =========================
# VALIDATION
# =========================
y_pred_rf = rf_model.predict(X_valid)

print("Validation Accuracy:", accuracy_score(y_valid, y_pred_rf))
print(classification_report(y_valid, y_pred_rf))

# =========================
# TEST PREDICTIONS
# =========================
print("Predicting test set...")
test_pred_rf = rf_model.predict(X_test_final)

# =========================
# SAVE SUBMISSION
# =========================
submission_rf = pd.DataFrame({
    'id': test_ids,
    'Irrigation_Need': test_pred_rf
})

submission_rf.to_csv('submission_rf.csv', index=False)
print("submission_rf.csv created successfully")
print(submission_rf.head())

Training Random Forest...
Training complete.
Validation Accuracy: 0.972468253968254
              precision    recall  f1-score   support

        High       0.99      0.50      0.67      4202
         Low       0.99      1.00      0.99     73983
      Medium       0.95      0.98      0.96     47815

    accuracy                           0.97    126000
   macro avg       0.97      0.83      0.87    126000
weighted avg       0.97      0.97      0.97    126000

Predicting test set...
submission_rf.csv created successfully
       id Irrigation_Need
0  630000             Low
1  630001             Low
2  630002             Low
3  630003             Low
4  630004             Low


# LightGBM Model

In [13]:
pip install lightGBM

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
print(y.value_counts())

Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64


In [49]:
print(train_df.columns.tolist())
print(X.columns.tolist())
print(test_df.columns.tolist())

['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region', 'Irrigation_Need']
['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Field_Area_hectare', 'Mulching_Used', 'Region', 'Irrigation_Need']
['id', 'Soil_Type', 'Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Field_Area_hectare', 'Mulching_Used', 'Previous_Irrigation_mm', 'Region']


In [66]:
# =========================================
# IMPORTS
# =========================================
import pandas as pd
import numpy as np
import optuna

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# =========================================
# LOAD DATA
# =========================================
train_df = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\train.csv')
test_df = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\test.csv')

# =========================================
# FEATURES / TARGET
# =========================================
X = train_df.drop(columns=[
    'id',
    'Irrigation_Need',
    'Irrigation_Type',
    'Water_Source',
    'Previous_Irrigation_mm'
]).copy()

y = train_df['Irrigation_Need'].copy()

X_test = test_df.drop(columns=[
    'id',
    'Irrigation_Type',
    'Water_Source',
    'Previous_Irrigation_mm'
]).copy()

# =========================================
# CATEGORICAL COLUMNS
# =========================================
cat_cols = [
    'Soil_Type',
    'Crop_Type',
    'Crop_Growth_Stage',
    'Season',
    'Mulching_Used',
    'Region'
]

for col in cat_cols:
    X[col] = X[col].astype('category')
    X_test[col] = X_test[col].astype('category')
    X_test[col] = pd.Categorical(X_test[col], categories=X[col].cat.categories)

# =========================================
# CROSS VALIDATION
# =========================================
kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# =========================================
# OPTUNA OBJECTIVE
# =========================================
def objective(trial):
    params = {
        'objective': 'multiclass',
        'n_estimators': trial.suggest_int('n_estimators', 200, 400),
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.15),
        'num_leaves': trial.suggest_int('num_leaves', 20, 60),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 40),
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'class_weight': 'balanced',
        'random_state': 42,
        'verbosity': -1
    }

    scores = []

    for train_idx, valid_idx in kf.split(X, y):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        for col in cat_cols:
            X_train[col] = X_train[col].astype('category')
            X_valid[col] = pd.Categorical(
                X_valid[col],
                categories=X_train[col].cat.categories
            )

        model = LGBMClassifier(**params)
        model.fit(X_train, y_train)

        preds = model.predict(X_valid)
        score = f1_score(y_valid, preds, average='weighted', zero_division=0)
        scores.append(score)

    return np.mean(scores)

# =========================================
# RUN OPTUNA
# =========================================
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

print('Best Parameters:')
print(study.best_params)
print('Best CV Score:', study.best_value)

# =========================================
# TRAIN FINAL MODEL
# =========================================
best_model = LGBMClassifier(
    objective='multiclass',
    **study.best_params,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    verbosity=-1
)

best_model.fit(X, y)

# =========================================
# PREDICT TEST
# =========================================
test_preds = best_model.predict(X_test)

[I 2026-04-24 08:10:45,588] A new study created in memory with name: no-name-8a97288d-a88d-4b07-9512-d126018d90cd
[I 2026-04-24 08:13:06,420] Trial 0 finished with value: 0.982980303284065 and parameters: {'n_estimators': 203, 'learning_rate': 0.1075608737995761, 'num_leaves': 27, 'min_child_samples': 20}. Best is trial 0 with value: 0.982980303284065.
[I 2026-04-24 08:16:18,570] Trial 1 finished with value: 0.9833471711028855 and parameters: {'n_estimators': 322, 'learning_rate': 0.1445386003334378, 'num_leaves': 46, 'min_child_samples': 28}. Best is trial 1 with value: 0.9833471711028855.
[I 2026-04-24 08:18:53,536] Trial 2 finished with value: 0.9840827429576482 and parameters: {'n_estimators': 246, 'learning_rate': 0.13760936329812995, 'num_leaves': 58, 'min_child_samples': 31}. Best is trial 2 with value: 0.9840827429576482.
[I 2026-04-24 08:21:36,023] Trial 3 finished with value: 0.984051105052774 and parameters: {'n_estimators': 261, 'learning_rate': 0.14076396353613507, 'num_le

Best Parameters:
{'n_estimators': 246, 'learning_rate': 0.13760936329812995, 'num_leaves': 58, 'min_child_samples': 31}
Best CV Score: 0.9840827429576482


In [67]:
print("Accuracy:", np.mean(cv_results['test_accuracy']), "+/-", np.std(cv_results['test_accuracy']))
print("Balanced Accuracy:", np.mean(cv_results['test_balanced_accuracy']), "+/-", np.std(cv_results['test_balanced_accuracy']))
print("Precision:", np.mean(cv_results['test_precision']), "+/-", np.std(cv_results['test_precision']))
print("Recall:", np.mean(cv_results['test_recall']), "+/-", np.std(cv_results['test_recall']))
print("F1 Score:", np.mean(cv_results['test_f1']), "+/-", np.std(cv_results['test_f1']))

Accuracy: 0.9820634920634921 +/- 0.0002694207741432371
Balanced Accuracy: 0.9692689128272534 +/- 0.0010440583299921068
Precision: 0.9823333961551638 +/- 0.00025495108130862976
Recall: 0.9820634920634921 +/- 0.0002694207741432371
F1 Score: 0.982109800649836 +/- 0.00026280097237672167


In [68]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Irrigation_Need': test_preds
})

submission.to_csv(r'C:\Users\prchandr\Downloads\submission_lgbm.csv', index=False)
print('submission_lgbm.csv created successfully.')

submission_lgbm.csv created successfully.


# CatBoost Model

In [75]:
# =========================================
# IMPORTS
# =========================================
import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

# =========================================
# LOAD DATA
# =========================================
train_df = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\train.csv')
test_df = pd.read_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\test.csv')

# =========================================
# FEATURES / TARGET
# =========================================
X = train_df.drop(columns=[
    'id',
    'Irrigation_Need',
    'Irrigation_Type',
    'Water_Source',
    'Previous_Irrigation_mm'
]).copy()

y = train_df['Irrigation_Need'].copy()

X_test = test_df.drop(columns=[
    'id',
    'Irrigation_Type',
    'Water_Source',
    'Previous_Irrigation_mm'
]).copy()

# =========================================
# CATEGORICAL COLUMNS
# =========================================
cat_cols = [
    'Soil_Type',
    'Crop_Type',
    'Crop_Growth_Stage',
    'Season',
    'Mulching_Used',
    'Region'
]

# make sure they exist
cat_cols = [col for col in cat_cols if col in X.columns]

# CatBoost can use string/object categorical columns directly
for col in cat_cols:
    X[col] = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

# optional missing-value safety
for col in cat_cols:
    X[col] = X[col].fillna('Missing')
    X_test[col] = X_test[col].fillna('Missing')

# indices of categorical columns for CatBoost
cat_feature_indices = [X.columns.get_loc(col) for col in cat_cols]

# =========================================
# CROSS VALIDATION
# =========================================
kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

cv_scores = []

for train_idx, valid_idx in kf.split(X, y):
    X_train = X.iloc[train_idx].copy()
    X_valid = X.iloc[valid_idx].copy()
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        loss_function='MultiClass',
        eval_metric='TotalF1',
        iterations=300,
        learning_rate=0.1,
        depth=6,
        random_seed=42,
        verbose=0
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_feature_indices
    )

    preds = model.predict(X_valid)
    preds = preds.flatten()

    score = f1_score(y_valid, preds, average='weighted', zero_division=0)
    cv_scores.append(score)

print('CatBoost CV Weighted F1:', np.mean(cv_scores))

# =========================================
# TRAIN FINAL MODEL ON FULL DATA
# =========================================
final_model = CatBoostClassifier(
    loss_function='MultiClass',
    eval_metric='TotalF1',
    iterations=300,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    verbose=0
)

final_model.fit(
    X,
    y,
    cat_features=cat_feature_indices
)

# =========================================
# PREDICT TEST
# =========================================
test_preds = final_model.predict(X_test)
test_preds = test_preds.flatten()

CatBoost CV Weighted F1: 0.9846180222515225


In [61]:
final_model.fit(
    X,
    y,
    cat_features=cat_feature_indices
)

test_preds = final_model.predict(X_test)
test_preds = test_preds.flatten()

In [62]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'Irrigation_Need': test_preds
})

submission.to_csv(r'C:\Users\prchandr\Downloads\playground-series-s6e4 (1)\submission_catboost.csv', index=False)
print('submission_catboost.csv created successfully.')

submission_catboost.csv created successfully.


# Discussion of Model Approach

I initially started with a basic Gradient Boosting model which performed poorly with a weighted score of around 0.58 so I decided to try creating interaction features like soil moisture, temperature, humidity, and rainfall. 

Then, I focused on tuning the Gradient Boosting model which took a very long time (the model took 311 minutes to run). As expected, the model performed much better after this step. The best performing setup was approximately n_estimators = 200, learning_rate = 0.1, max_depth = 4, subsample = 0.8, which achieved a weighted F1 score of about 0.96. I think this worked well because a max_depth of 4 was enough to capture interactions like low soil moisture combined with high temperature without overfitting, while increasing n_estimators to 200 gave the model enough iterations to refine the model.

Even though Gradient Boosting improved signficantly from before (0.58 before and 0.96 after), it was still very slow compared to LightGBM (30 minutes) and CatBoost (30 minutes). Both of those models reached similar or slightly better performance and was much faster. LightGBM still performed the best overall and was much more quicker to tune. In terms of leaderboard performance, all three models ended up relatively close, but LightGBM had the best combination of speed and accuracy, while Gradient Boosting was the most computationally expensive. I was very shocked to find out that Gradient Boosting could be optimized through feature engineering and was not expecting such an improvement.

Afterwards, I tested LightGBM and CatBoost which both performed very well above 0.95, and required significantly less tuning compared to Gradient Boosting.I liked CatBoost the most because it required the least amount of transformation while performing similarly to LightGBM. However, LightGBM performed the best and brought me up on the leaderboard to 1333 (compared to the 3000s earlier). I think it performed the best due to the numeric values in the dataset.

Additionally, when calculating the performance metrics this time I used a weighted F1 score to properly evaluate performance across multiple irrigation classes to account for the class imbalance. For my next steps, I would like to try combining my best models using ensembling or using a simple stacking approach, since they capture slightly different patterns and could improve overall performance. I would also refine my feature engineering by using feature importance from LightGBM to selectively add only the most impactful interaction features instead of adding many at once. 